This notebook uses the expanded dataset prepared by `preprocessing_refined.ipynb`: train on 2018–2024 and test on the held-out 2025 season. Run refined preprocessing before classification. The data-loading cell checks that both datasets contain the expected seasons.


In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F

from classification_helper import assemble, perform, print_results, setup_spark

In [ ]:
spark = setup_spark("Classification_nopitch")

selected_types = ["FF", "SI", "SL", "CH", "FC", "ST", "CU", "FS", "KC"]
train = spark.read.parquet("./data/temporal_reduced_traintest/train")
test = spark.read.parquet("./data/temporal_reduced_traintest/test")

train_years = {row.game_year for row in train.select("game_year").distinct().collect()}
test_years = {row.game_year for row in test.select("game_year").distinct().collect()}
assert train_years == set(range(2018, 2025)), "Training data must contain the 2018-2024 seasons"
assert test_years == {2025}, "Test data must contain only the held-out 2025 season"

train = train.filter(F.col("pitch_type").isin(selected_types))
test = test.filter(F.col("pitch_type").isin(selected_types))

print("Training observations:", train.count())
print("Test observations:", test.count())
train.groupBy("game_year").count().orderBy("game_year").show()
test.groupBy("game_year").count().orderBy("game_year").show()


In [ ]:
label_indexer = StringIndexer(inputCol="pitch_type", outputCol="label")
idx_model = label_indexer.fit(train)
train = idx_model.transform(train)
test = idx_model.transform(test)

In [ ]:
numerical = [
    "on_3b", "on_2b", "on_1b", "inning", "sz_top", "sz_bot",
    "bat_score", "fld_score", "bat_score_diff", "home_win_exp",
    "bat_win_exp", "age_pit", "age_bat", "n_thruorder_pitcher",
    "n_priorpa_thisgame_player_at_bat", "pitcher_days_since_prev_game",
    "batter_days_since_prev_game"
]
one_hot_encoded = [column for column in train.columns if column.endswith("_idx")]
features = numerical + one_hot_encoded

train, test = assemble(train, test, features)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(featuresCol="features", maxIter=100),
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features", labelCol="label", maxDepth=10, seed=42
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=100, maxDepth=10, seed=42
    )
}

In [ ]:
metrics = ["accuracy", "f1", "weightedPrecision", "weightedRecall"]
results = perform(train, test, models, metrics, "label")
predictions = {
    name: result["predictions"]
    for name, result in results.items()
}
print_results(results)

In [ ]:
decision_tree_predictions = predictions["Decision Tree"]
pred_and_labels = (
    decision_tree_predictions.select("prediction", "label")
    .rdd.map(lambda row: (float(row.prediction), float(row.label)))
)
class_metrics = MulticlassMetrics(pred_and_labels)
for index, pitch in enumerate(idx_model.labels):
    print(
        f"{pitch:25s} Precision: {class_metrics.precision(float(index)):.3f} | "
        f"Recall: {class_metrics.recall(float(index)):.3f} | "
        f"F1: {class_metrics.fMeasure(float(index)):.3f}"
    )

macro_f1 = sum(
    class_metrics.fMeasure(float(index))
    for index in range(len(idx_model.labels))
) / len(idx_model.labels)
print(f"Macro F1: {macro_f1:.4f}")

for name, model_predictions in predictions.items():
    model_pairs = (
        model_predictions.select("prediction", "label")
        .rdd.map(lambda row: (float(row.prediction), float(row.label)))
    )
    model_metrics = MulticlassMetrics(model_pairs)
    model_macro_f1 = sum(
        model_metrics.fMeasure(float(index))
        for index in range(len(idx_model.labels))
    ) / len(idx_model.labels)
    print(f"{name} Macro F1: {model_macro_f1:.4f}")